# GCI Home Credit — candidata v13

A melhor referência pública informada pela participante é **0,77820**, em
**23º/1949**. A nota pública desta candidata ainda não foi medida.

Esta rodada testa modelos por pares, CatBoost com novas taxas de precificação,
TabM com embeddings numéricos, EBM e combinações aprendidas. O pacote mantém
o código dos testes e utiliza apenas os componentes selecionados na receita.

O ganho local é pequeno; não há evidência de que a candidata alcance 0,80 público.


## Executar e preservar a referência

Extraia o ZIP completo e mantenha sua estrutura de pastas. Coloque os dados
fornecidos em `Competition/input`, ou ajuste `INPUT_DIR` abaixo. Execute as
células em ordem. As dependências, os scripts e os checkpoints são conferidos
antes de reconstruir a submissão.

O CSV para testar é `output_v13/submission_v13.csv`. O backup
`submission_077820_backup.csv` preserva exatamente o arquivo de **0,77820**.
Pelas regras fornecidas, **a última submissão conta**. Se a candidata piorar,
reenvie o backup.


In [1]:
import os, sys, json, subprocess, importlib.metadata
from pathlib import Path
os.environ.setdefault('OMP_NUM_THREADS','4')
os.environ.setdefault('OPENBLAS_NUM_THREADS','4')
PINS = {'interpret-core': '0.7.8', 'lightgbm': '4.7.0', 'numpy': '2.3.5', 'pandas': '2.2.3', 'rtdl-num-embeddings': '0.0.12', 'scikit-learn': '1.8.0', 'scipy': '1.17.0', 'tabm': '0.0.3', 'torch': '2.14.1+cpu'}
recipe=json.loads(Path('v13_recipe.json').read_text())
selected=[n for n,w in recipe['weights'].items() if n!='reference_v12' and w>0]
needed={'numpy','pandas','scipy','scikit-learn','lightgbm'}
if any(n.startswith('ebm') for n in selected):needed.add('interpret-core')
if any(n.startswith('tabm') for n in selected):needed.update(['torch','tabm','rtdl-num-embeddings'])
if any(n.startswith(('pair','xgb')) for n in selected):needed.add('xgboost')
if any(n.startswith('cat') for n in selected):needed.add('catboost')
missing=[]
for name in sorted(needed):
    try:version=importlib.metadata.version(name)
    except importlib.metadata.PackageNotFoundError:version=None
    if version!=PINS[name]:missing.append(name)
if 'torch' in missing:
    subprocess.run([sys.executable,'-m','pip','install','torch=='+PINS['torch'],
                    '--index-url','https://download.pytorch.org/whl/cpu'],check=True)
    missing.remove('torch')
if missing:
    subprocess.run([sys.executable,'-m','pip','install',*[p+'=='+PINS[p] for p in missing]],check=True)
print('Versões:',{p:importlib.metadata.version(p) for p in sorted(needed)})


Versões: {'interpret-core': '0.7.8', 'lightgbm': '4.7.0', 'numpy': '2.3.5', 'pandas': '2.2.3', 'rtdl-num-embeddings': '0.0.12', 'scikit-learn': '1.8.0', 'scipy': '1.17.0', 'tabm': '0.0.3', 'torch': '2.14.1+cpu'}


In [2]:
INPUT_DIR=Path(os.environ.get('HOME_CREDIT_INPUT','Competition/input'))
REFERENCE_DIR=Path('reference_v12')
OUTPUT_DIR=Path('output_v13')
RETRAIN_NEW_COMPONENTS=False
assert (INPUT_DIR/'train.csv').is_file(), 'Ajuste INPUT_DIR para os CSVs fornecidos.'
from deliver_v13 import run
report=run(INPUT_DIR,REFERENCE_DIR,OUTPUT_DIR,Path('v13_recipe.json'),retrain=RETRAIN_NEW_COMPONENTS)


Reusing output_v13/tabmple_fold0.npz
Reusing output_v13/tabmple_fold1.npz
Reusing output_v13/tabmple_fold2.npz
Reusing output_v13/tabmple_fold3.npz
Reusing output_v13/tabmple_fold4.npz
{
  "metrics": {
    "all": {
      "reference_auc": 0.7737929598769375,
      "candidate_auc": 0.7739951817519151,
      "delta": 0.0002022218749775373,
      "rows": 171202
    },
    "selection_01": {
      "reference_auc": 0.7727768508062283,
      "candidate_auc": 0.7728845771498388,
      "delta": 0.0001077263436105591,
      "rows": 68482
    },
    "additional_234": {
      "reference_auc": 0.7744708326945058,
      "candidate_auc": 0.7747320948784864,
      "delta": 0.00026126218398059287,
      "rows": 102720
    }
  },
  "csv": "output_v13/submission_v13.csv",
  "sha256": "f145b9af134172104217d5f0910b8e3423a806a67e340c4abd1d4955a28beb95"
}


In [3]:
import numpy as np
import pandas as pd
from experiment_v13 import sha
csv_path=OUTPUT_DIR/'submission_v13.csv'
submission=pd.read_csv(csv_path,float_precision='round_trip')
sample=pd.read_csv(INPUT_DIR/'sample_submission.csv')
assert list(submission.columns)==['SK_ID_CURR','TARGET']
assert np.array_equal(submission.SK_ID_CURR,sample.SK_ID_CURR)
assert len(submission)==61500 and submission.SK_ID_CURR.is_unique
assert submission.TARGET.between(0,1).all()
assert sha(csv_path)==report['candidate_csv_sha256']
assert sha('submission_077820_backup.csv')==report['reference_csv_sha256']
print('CSV conferido:',len(submission),'linhas; IDs, ordem e valores válidos.')
print('SHA256:',sha(csv_path))
print('Pesos:',report['recipe']['weights'])
print(pd.DataFrame(report['metrics']).T.to_string())
print('Referência pública: 0.77820; candidata pública: ainda não medida.')


CSV conferido: 61500 linhas; IDs, ordem e valores válidos.
SHA256: f145b9af134172104217d5f0910b8e3423a806a67e340c4abd1d4955a28beb95
Pesos: {'reference_v12': 0.8, 'tabmple': 0.15, 'ebm12': 0.05}
                reference_auc  candidate_auc     delta      rows
all                  0.773793       0.773995  0.000202  171202.0
selection_01         0.772777       0.772885  0.000108   68482.0
additional_234       0.774471       0.774732  0.000261  102720.0
Referência pública: 0.77820; candidata pública: ainda não medida.


## Validação e reprodução

Os modelos usam cinco folds estratificados com seed 42. As transformações que
usam `TARGET` são calculadas com rótulos do treino de cada fold. As taxas de
precificação do treino usam cinco folds internos com seed 17.

Modelos e pesos são triados nos folds 0/1. Os três folds adicionais verificam
consistência; decisões de modelos anteriores e o early stopping das redes usam
validação. Isso não constitui um holdout independente de todo o processo e não
garante melhora pública.

Os componentes anteriores são reutilizados dos arquivos do pacote v12, incluído
em `reference_v12`. Seus notebooks preservam o código dos treinos anteriores.
Os novos componentes selecionados desta rodada foram treinados nos cinco folds.
Use `RETRAIN_NEW_COMPONENTS=True` para repeti-los em uma pasta nova; os antigos
checkpoints ficam disponíveis. As variáveis auxiliares anteriores são reutilizadas.

Não foram usados dados externos, pesos pré-treinados ou ajustes manuais do risco
de linhas do teste. Referências de algoritmos:

- [TabM](https://github.com/yandex-research/tabm)
- [Embeddings numéricos](https://github.com/yandex-research/rtdl-num-embeddings)
- [InterpretML / EBM](https://interpret.ml/docs/ebm.html)
- [XGBoost ranking](https://xgboost.readthedocs.io/en/stable/tutorials/learning_to_rank.html)
- README e tutorial GCI fornecidos; referências da versão anterior nos notebooks incluídos.

Para executar pelo terminal:

```bash
python deliver_v13.py --input Competition/input --reference reference_v12 --output output_v13 --recipe v13_recipe.json
```

Acrescente `--retrain` para repetir os novos componentes selecionados.
